# **11766 Advanced Perception for Mobile Robotics**
## Master's Degree in Intelligent Systems
### University of the Balearic Islands
---

##### Write in the following the names of the members of the group:
- John Smith 1
- John Smith 2

## **Instructions**

- **Do not delete any of the provided cells or functions**. Write your code where it is indicated. You may add new cells or functions as needed to complete your work.
- Along with this assignment, please submit a comprehensive **report** in PDF format explaining your implemented solutions. This report can include, for instance:
    - A technical explanation: Clearly describe the algorithms and techniques used, including relevant code snippets.
    - Results: Present your results in a clear and concise manner. Use visualizations such as graphs or tables to enhance understanding.
    - Analysis: Interpret your results and discuss their significance. Consider, for instance, the following questions:
        - Are the results as expected? Why or why not?
        - What factors might have influenced the results?
    - Conclusions: Summarize your findings and provide overall conclusions about your project.


# **EKF Localization**

In this assignment you need to implement an EKF algorithm for localizing a robot in a given landmark map. 
The data for this exercise is recorded on a differential drive robot equipped with a sensor able to detect the distance and the angle of landmarks (e.g., beacons). After executing the following cell, the figure below visualizes the landmark map and the actual trajectory (ground truth) taken by the robot.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
from utils import *
%matplotlib inline

# Read dataset
data = pickle.load(open("data/dataset_2d_landmarks.p", "rb"))

# Get landmark coordinates 
M = data['M']

# Get ground truth trajectory
gt_traj = data['gt']

# Show map
plt.figure()
plt.plot(M[:,0], M[:,1], '^r')

# Show ground truth trajectory
for i in range(0,len(gt_traj), 10):
    plt.plot(gt_traj[i][0],gt_traj[i][1], '.b')

plt.show()

The following data is provided in `data`:

- *M* is the map of the environment where the robot must be localized, i.e.: the true coordinates of the landmarks in map frame.
- *gt* is the ground truth trajectory (true poses in the map frame). You may want to use them for checking your results.
- *odom* are the noisy odometry readings observed by the robot during navigation in the form: $\hat{x}_t,\hat{y}_t, \hat{\theta}_t$ in the odometry frame.
- *z* are the sensor measurements. Each measurement $z_t$, corresponding to the time step $t$, contains a set of observed landmarks $[\rho_i, \phi_i, id_i]$, where $\rho_i$ is the measured distance, $\phi_i$ is the measured angle, and $id_i$ is the id of the observed landmark. Note that this id corresponds to the index of the landmark in *M*.

You can access the `data` as follows:

In [ ]:
# Get odomety at timestamp 10
odom_10 = data['odom'][10]
print("Odom at step 10 is: \n", odom_10)

# get observation at timestamp 10
z_10 = data['z'][10]
print("Observation at step 10 is: \n", z_10)

## **Prediction**

The `ekf_predict` function computes a prediction about the robot's pose after moving by using the odometry motion model.

It takes as input:

- The current belief about the pose of the robot, represented as a Gaussian distribution $\mathcal{N}(\mu_{t-1},\Sigma_{t-1})$ 
- The odometry readings $u_t$, which is a list where $u_t[0]$ and $u_t[1]$ are the odometry readings at the previous time step ($t - 1$) and the current time step ($t$), respectively.
- The process noise $R_t$.

The output of this function is a prediction about the robot's pose in the next time step $t$, $\mathcal{N}(\overline{\mu}_{t},\overline{\Sigma}_{t})$. You are also provided with a function called `inv_motion_model` to compute $u_t = [\delta_{rot1}, \delta_{trans}, \delta_{rot2}]$ between two consecutive odometry readings.

**Tasks:**
- Implement the `ekf_predict` function in the `utils.py` file and verify if it is correct with the following cell. 

In [ ]:
mu_t = np.array([2, 2, np.pi/2]).reshape(3, 1)
S_t = np.array([[1, 0, 0],[0, 1, 0], [0, 0, 1]])
R_t = np.zeros((3,3))
print('Before prediction')
print(mu_t)
print(S_t)
print('---')

u_p, S_p = ekf_predict(mu_t, S_t, np.array([[0,0,0],[1,0,0]]), R_t)

print('After prediction')
print(u_p)
print(S_p)

## **Correction**

The `ekf_correct` implements the correction step of the EKF that corrects the prediction according to the sensor measurements.

It takes as input:

- The current prediction about the pose of the robot represented as a Gaussian distribution $\mathcal{N}(\overline{\mu}_{t},\overline{\Sigma}_{t})$
- The sensor measurements $z_t$. Remember that it each sensor measurement includes observations of several landmarks.
- The measurement noise $Q_t$.
- The map $M$.

The output is the new belief about the robot's pose $\mathcal{N}({\mu}_{t},{\Sigma}_{t})$.

**Tasks:**
- Implement the `ekf_correct` function in the `utils.py` file and verify that it is correct with the following cell. Use the provided function `wrapToPi` to ensure that each angle is between $-\pi$ and $\pi$ everywhere you deal with it.

In [ ]:
# 3x3 Process noise
sigma_x = 0.25  # [m]
sigma_y = 0.25  # [m]
sigma_theta = np.deg2rad(10)  # [rad]
R = np.diag(np.array([sigma_x, sigma_y, sigma_theta])**2)

# 2x2 Observation noise
sigma_r = 0.1  # [m]
sigma_phi = np.deg2rad(5)  # [rad]
Q = np.diag(np.array([sigma_r, sigma_phi])**2)

# initial state
mu_bar = np.array([2, 2, np.pi/2]) 
S_bar = np.array([[1, 0, 0],[0, 1, 0], [0, 0, np.pi/3]])

for i in range(1,100):   
    u_t = [data['odom'][i-1], data['odom'][i]]
    mu_bar, S_bar = ekf_predict(mu_bar.reshape(3,1), S_bar, u_t, R)

print("Only running predict")
print(mu_bar)
print(S_bar)
print('---')

mu = np.array([2, 2, np.pi/2]) 
S = np.array([[1, 0, 0],[0, 1, 0], [0, 0, np.pi/3]])

for i in range(1,100):   
    u_t = [data['odom'][i-1],data['odom'][i]]
    z = data['z'][i]
    mu_bar, sigma_bar = ekf_predict(mu.reshape(3,1), S, u_t, R)
    mu, S = ekf_correct(mu_bar, sigma_bar, z, Q, M)
print("Running predict and correct")
print(mu)
print(S)
print('---')

print('Ground Truth Pose at 100')
print(gt_traj[100].reshape(3,1))

## **Localization**

**Tasks:**

- Once you have completed all the above functions, implement the main procedure of an EKF localization algorithm, which recursively estimates the pose of the robot using the odometry data and the sensor measurements.

Assume that the initial belief at time $t=0$ is:

$\mu = [2, 2, \pi/2]'$\
$
\Sigma = \left(\begin{array}{cc} 
1 & 0 & 0\\
0 & 1 & 0 \\
0 & 0 & \pi/3
\end{array}\right)
$ 
            
The process noise $R$ and measurement noise $Q$ are defined, respectively, as:\
$R = \left(\begin{array}{cc} 
\sigma_x^2 & 0 & 0 \\
0 & \sigma_y^2 & 0 \\
0 & 0 &  \sigma_{theta}^2
\end{array}\right)
$

with $\sigma_x = 0.25$ meters, $\sigma_y = 0.25$ meters and $\sigma_{theta} = 10$ degrees, and 
 
$Q = 
\left(\begin{array}{cc} 
\sigma_r^2 & 0 \\
0 & \sigma_{phi}^2 
\end{array}\right)
$

with $\sigma_r = 0.10$ meters, $\sigma_{phi} = 5$ degrees. 

- Plot the estimated trajectory, along with the ground truth and the map, after processing the whole sequence.

In [ ]:
# 3x3 process noise
sigma_x = 0.25  # [m]
sigma_y = 0.25  # [m]
sigma_theta = np.deg2rad(10)  # [rad]
R = np.diag(np.array([sigma_x, sigma_y, sigma_theta])**2)

# 2x2 observation noise
sigma_r = 0.1  # [m]
sigma_phi = np.deg2rad(5)  # [rad]
Q = np.diag(np.array([sigma_r, sigma_phi])**2)

# initial state
mu = np.array([2, 2, np.pi/2]) 
S = np.array([[1, 0, 0],[0, 1, 0], [0, 0, np.pi/3]])

# visualize
plot_state(mu, S, M)

In [ ]:
# Write here the code to process the full trajectory

# YOUR CODE HERE
raise NotImplementedError()
# -----